# Single-machine confinement scaling and dimensional analysis

A reference study (#1621) of what a single machine can and cannot say about
confinement scaling, on VEST's Tier A table (#548) as the worked case. It asks,
in order:

1. Which engineering exponents does the data identify?
2. How robust are they to estimator, grouping and measurement error?
3. Which dimensionless groups follow from dimensional analysis alone?
4. Where does the Connor–Kadomtsev constraint come from, and what does it add?
5. When are engineering and dimensionless regressions the same model?
6. When they disagree, which of four failure modes is it?

The four failure modes are kept apart throughout:

| | failure | what it means |
|---|---|---|
| **A** | rank / identifiability | the data do not span the predictor directions |
| **B** | ill-conditioned transformation | exponents identified, but the map to dimensionless indices amplifies their error |
| **C** | similarity-constraint failure | the fitted law is incompatible with the Connor–Kadomtsev assumption |
| **D** | definition mismatch | inconsistent definitions of density, power, geometry or derived groups |

Every computation is a VAFT API: `vaft.process.confinement` for inference and
`vaft.formula.dimensional` for the exact dimensional algebra. The experimental
confinement result and the published-scaling comparison stay in
`confinement_time_scaling.ipynb`.

In [ ]:
from fractions import Fraction

import numpy as np
import pandas as pd

from vaft.data.public import load_vest_tier_a_confinement
from vaft.formula import dimensional as dim
from vaft.formula.equilibrium import kadomtsev_constraint_from_engineering_exponents
from vaft.process import confinement as conf

pd.set_option("display.width", 140)
table = load_vest_tier_a_confinement("primary")
shots = table["shot"].to_numpy()
tau = table["tau_e_th_s"].to_numpy(float)
engineering = {"i_p": table["i_p_A"].to_numpy(float), "b_t": table["b_t_T"].to_numpy(float),
               "p_net": table["p_loss_W"].to_numpy(float)}
print(f"VEST Tier A primary selection: {len(table)} states over {table['shot'].nunique()} shots "
      f"({table.attrs.get('selection', 'primary')})")

VEST Tier A primary selection: 59 states over 19 shots (primary)


## Stage I — the engineering fit and what identifies it

$\tau_E = C\, I_p^{\alpha_I} B_T^{\alpha_B} P_L^{\alpha_P}$, errors clustered by
shot. Size exponents are *not* fitted: VEST's shot-to-shot geometry variation is
plasma position, not machine size.

In [ ]:
fit = conf.fit_confinement_scaling(tau, engineering, shots)
huber = conf.fit_confinement_scaling(tau, engineering, shots, robust=True)
boot = conf.bootstrap_confinement_scaling(tau, engineering, shots, n_boot=2000, seed=1621)
loso = conf.leave_one_group_out_scaling(tau, engineering, shots)
rows = []
for j, name in enumerate(fit.names):
    lo, hi = boot["percentile95"][j] if name in boot["names"] else (np.nan, np.nan)
    rows.append({"term": name, "estimate": fit.coef[j], "se (by shot)": fit.stderr[j],
                 "se (iid)": fit.stderr_iid[j], "bootstrap 95%": f"[{lo:.2f}, {hi:.2f}]",
                 "Huber": huber.coef[j]})
print(pd.DataFrame(rows).round(3).to_string(index=False))
print(f"\nn = {fit.n}, shots = {fit.n_groups}, RMSE(log) = {fit.rmse_log:.3f}, "
      f"leave-one-shot-out RMSE(log) = {loso['rmse_log']:.3f}")
print(f"max leverage {fit.leverage.max():.2f}, max Cook's distance {np.nanmax(fit.cooks_distance):.2f}")

 term  estimate  se (by shot)  se (iid)    bootstrap 95%   Huber
log_C   -16.530         2.214     1.448 [-20.75, -11.16] -18.274
  i_p     1.952         0.304     0.218     [1.32, 2.57]   2.269
  b_t     0.610         0.313     0.269    [-0.62, 1.28]   0.828
p_net    -0.904         0.110     0.077   [-1.15, -0.71]  -1.025

n = 59, shots = 19, RMSE(log) = 0.174, leave-one-shot-out RMSE(log) = 0.202
max leverage 0.20, max Cook's distance 0.35


### Identifiability: correlation, VIF, rank, condition number, principal directions

`predictor_principal_directions` decomposes the standardised log design. A small
singular value is a combination of predictors the sample barely varies, so the
matching combination of exponents is poorly determined. Each direction's spread is
then split into the part carried by differences between shot means and the part
carried by evolution within shots (the two add in quadrature).

In [ ]:
ident = conf.assess_predictor_identifiability(engineering)
print("correlation of log predictors:")
print(pd.DataFrame(ident["correlation"]).round(2).to_string())
print(f"\nVIF {', '.join(f'{k} {v:.1f}' for k, v in ident['vif'].items())}; "
      f"rank {ident['rank']}; Belsley condition number {ident['condition_number']:.1f}")

pcs = conf.predictor_principal_directions(engineering, groups=shots)
frame = pd.DataFrame(pcs["directions"], columns=pcs["names"])
frame.insert(0, "singular value", pcs["singular_values"])
for key in ("between", "within"):
    frame[f"{key} shots"] = pcs[key]["projected"]
print("\nprincipal directions (standardised log coordinates), strongest first:")
print(frame.round(2).to_string(index=False))
print(f"effective rank {pcs['effective_rank']} of {len(pcs['names'])}")

correlation of log predictors:
        i_p   b_t  p_net
i_p    1.00  0.04   0.88
b_t    0.04  1.00   0.35
p_net  0.88  0.35   1.00

VIF i_p 7.0, b_t 1.8, p_net 7.9; rank 4; Belsley condition number 251.9

principal directions (standardised log coordinates), strongest first:
 singular value   i_p   b_t  p_net  between shots  within shots
          10.67  0.65  0.28   0.70          10.48          2.03
           7.50 -0.37  0.93  -0.03           6.83          3.10
           1.96 -0.66 -0.23   0.71           1.30          1.47
effective rank 3 of 3


The weakest direction is mostly $\ln P_L - \ln I_p$, the loop voltage at fixed
current: ohmic heating ties $P$ to $I_p$, the rank problem (mode **A**) behind the
correlated $\alpha_I$–$\alpha_P$ errors. The split columns show that both shot-to-shot
differences and the evolution inside shots contribute to it; neither alone would
pin $\alpha_P$.

What the $B_T$ spread measures, and how the two Tier A shot blocks differ in
density, from the same table:

In [ ]:
log_b, log_r = np.log(table["b_t_T"].to_numpy(float)), np.log(table["r_geo_m"].to_numpy(float))
b0r0 = np.exp(log_b + log_r)                 # B_T = |B0 R0| / R_geo: B0 R0 is the TF current
print(f"corr(ln B_T, ln R_geo) = {np.corrcoef(log_b, log_r)[0, 1]:.2f}; "
      f"B0R0 spread (max/min - 1) {b0r0.max() / b0r0.min() - 1:.1%}, R_geo spread {np.exp(np.ptp(log_r)) - 1:.1%}")
dens = table.groupby("tier_a_block")["n_e_line_avg_m3"].median()
print("median Thomson line density by block [m^-3]:", dens.map("{:.2e}".format).to_dict(),
      f"ratio {dens.max() / dens.min():.1f}")

corr(ln B_T, ln R_geo) = -0.83; B0R0 spread (max/min - 1) 17.0%, R_geo spread 35.3%
median Thomson line density by block [m^-3]: {'399xx-403xx': '5.39e+18', '429xx-430xx': '1.44e+19'} ratio 2.7


### Errors in variables

$\tau_E = W/P$ couples the response to $P$, so the errors-in-variables check fits
$W$ against $P_{OH}$ (which does not use $W$) by orthogonal distance regression,
then reads $\alpha_P = \alpha_P^W - 1$. The assumed log errors are stated.

In [ ]:
odr = conf.fit_confinement_scaling_odr(
    table["w_th_J"].to_numpy(float),
    {"i_p": engineering["i_p"], "b_t": engineering["b_t"], "p_ohm": table["p_ohm_W"].to_numpy(float)},
    sigma_log_response=0.2, sigma_log_predictors={"i_p": 0.03, "b_t": 0.03, "p_ohm": 0.10})
w = dict(zip(odr["names"], odr["coef"]))
print(f"ODR on W: a_I {w['i_p']:.2f}, a_B {w['b_t']:.2f}, a_P {w['p_ohm'] - 1:.2f} (tau_E = W/P), "
      f"n = {odr['n']}; OLS on the same rows: a_P {dict(zip(odr['names'], odr['ols_coef']))['p_ohm'] - 1:.2f}")

ODR on W: a_I 2.16, a_B 0.85, a_P -0.98 (tau_E = W/P), n = 59; OLS on the same rows: a_P -0.94


## Stage II — Buckingham-Pi groups, exactly

For the collisional, finite-$\beta$ problem the variables are the state
$(n, T, B, R)$, the time $\tau$ and the constants $e$, $m_i$, $\mu_0$,
$\epsilon_0$ (shape, $q$ and the ion species fixed). A monomial
$\prod_j x_j^{a_j}$ is dimensionless exactly when $D\mathbf a = 0$, so the
groups span $\ker D$. `vaft.formula.dimensional` builds $D$ and its null space
over the rationals; no floating-point tolerance decides the rank.

In [ ]:
variables = ("n", "T", "B", "R", "tau", "e", "m_i", "mu0", "eps0")
D = dim.dimension_matrix(variables)
print(pd.DataFrame(D, index=dim.BASE_DIMENSIONS, columns=variables).astype(str).to_string())
basis = dim.dimensionless_groups(variables)
print(f"\n{len(variables)} variables - rank 4 = {len(basis)} groups; the null-space basis:")
print(pd.DataFrame([[str(v) for v in b] for b in basis], columns=variables).to_string())

    n   T   B  R tau  e m_i mu0 eps0
L  -3   2   0  1   0  0   0   1   -3
M   0   1   1  0   0  0   1   1   -1
T   0  -2  -2  0   1  1   0  -2    4
I   0   0  -1  0   0  1   0  -2    2

9 variables - rank 4 = 5 groups; the null-space basis:
   n  T   B  R tau   e m_i mu0 eps0
0  1  0   0  3   0   0   0   0    0
1  2  3  -3  0   3  -3   0   0    0
2  2  3   0  0   6   0  -3   0    0
3  1  1  -2  0   0   0   0   1    0
4  5  3  -6  0   6   0   0   0   -3


That basis is valid but not conventional, and not unique. The conventional
plasma groups are another basis of the same space:
$\Omega_i\tau_E = eB\tau/m_i$, $\rho_* = \sqrt{m_i T}/(eBR)$,
$\beta = \mu_0 n T/B^2$, $\nu_* \propto n e^4 R/(\epsilon_0^2 T^2)$, and
$(R/\lambda_D)^2 = n e^2 R^2/(\epsilon_0 T)$. Expressing each in the basis gives
an invertible matrix, an exact change of basis.

In [ ]:
half = Fraction(1, 2)
conventional = {
    "Omega_i tau_E": (0, 0, 1, 0, 1, 1, -1, 0, 0),
    "rho_*": (0, half, -1, -1, 0, -1, half, 0, 0),
    "beta": (1, 1, -2, 0, 0, 0, 0, 1, 0),
    "nu_*": (1, -2, 0, 1, 0, 4, 0, 0, -2),
    "(R/lambda_D)^2": (1, -1, 0, 2, 0, 2, 0, 0, -1),
}
for name, group in conventional.items():
    assert all(v == 0 for v in dim.monomial_dimension(group, variables)), name
change = pd.DataFrame({name: dim.express_in_basis(g, basis) for name, g in conventional.items()},
                      index=[f"Pi_{k + 1}" for k in range(len(basis))]).T
print(change.astype(str).to_string())
print(f"\ndeterminant {np.linalg.det(change.to_numpy(float)):.3g}: the five conventional groups are a basis")

                Pi_1  Pi_2  Pi_3 Pi_4 Pi_5
Omega_i tau_E      0  -1/3   1/3    0    0
rho_*           -1/3   1/3  -1/6    0    0
beta               0     0     0    1    0
nu_*             1/3  -4/3     0    0  2/3
(R/lambda_D)^2   2/3  -2/3     0    0  1/3

determinant -0.0185: the five conventional groups are a basis


## Stage III — from dimensional invariance to the Connor–Kadomtsev constraint

Dimensional analysis alone says $\Omega_i\tau_E = F(\rho_*, \beta, \nu_*,
R/\lambda_D)$. Over the varied state $(n, T, B, R)$ those four groups span all
four directions, so they impose **no** constraint on how $\tau_E$ may scale.

Connor and Taylor's similarity is a stronger, physical assumption: the
governing equations (Vlasov–Maxwell with Coulomb collisions) are taken in the
quasi-neutral limit, so the Debye group drops out. Three groups in a
four-dimensional state space leave one direction the law cannot have, the
constraint vector $\mathbf k$ with $\mathbf k \cdot \mathbf e_{\Omega\tau} = 0$.

In [ ]:
state = [0, 1, 2, 3]   # n, T, B, R
on_state = lambda g: [g[i] for i in state]  # noqa: E731
four = dim.similarity_constraint_vectors([on_state(conventional[k]) for k in ("rho_*", "beta", "nu_*", "(R/lambda_D)^2")])
three = dim.similarity_constraint_vectors([on_state(conventional[k]) for k in ("rho_*", "beta", "nu_*")])
print(f"with the Debye group:    {len(four)} constraints")
print(f"quasi-neutral (dropped): {len(three)} constraint, k = {[int(v) for v in three[0]]} on (n, T, B, R)"
      " (defined up to sign)")

with the Debye group:    0 constraints
quasi-neutral (dropped): 1 constraint, k = [8, 2, 5, -4] on (n, T, B, R) (defined up to sign)


An engineering law $\tau_E \propto I_p^{\alpha_I} B^{\alpha_B} P^{\alpha_P}
n^{\alpha_n} R^{\alpha_R}$ maps onto the state at fixed shape and $q$
($I_p \propto BR$, $P = W/\tau_E$, $W \propto nTR^3$), with every exponent
divided by $1+\alpha_P$. Applying $\mathbf k$ to $\Omega_i\tau_E$ and clearing
that factor gives exactly the engineering Kadomtsev residual
$4\alpha_R - 8\alpha_n - \alpha_I - 3\alpha_P - 5\alpha_B - 5$, derived here
from the declared assumption rather than inserted.

In [ ]:
k = np.array([float(v) for v in three[0]])
rng = np.random.default_rng(1621)
worst = 0.0
for _ in range(200):
    a_I, a_B, a_P, a_n, a_R = rng.uniform(-1.5, 2.0, 5)
    if abs(1 + a_P) < 0.05:
        continue
    x = np.array([float(v) for v in dim.state_exponents_from_engineering_exponents(a_I, a_B, a_P, a_n, a_R)])
    x[2] += 1.0                                   # Omega_i tau_E = e B tau / m_i
    from_similarity = -float(k @ x) * (1 + a_P)
    worst = max(worst, abs(from_similarity - kadomtsev_constraint_from_engineering_exponents(a_I, a_B, a_P, a_n, a_R)))
print(f"largest difference from the engineering Kadomtsev residual over 200 random laws: {worst:.1e}")

largest difference from the engineering Kadomtsev residual over 200 random laws: 7.1e-15


For a single machine the constraint cannot be *tested*: VEST does not vary $R$.
It can only *complete* the law, assigning
$\alpha_R^{KCT} = \tfrac14\alpha_I + \tfrac54\alpha_B + \tfrac34\alpha_P + 2\alpha_n + \tfrac54$,
which is an assumption, never a VEST measurement.

In [ ]:
alpha = fit.exponents()
names = ["i_p", "b_t", "p_net"]
cov = np.asarray(fit.cov)[1:, 1:]
ind = conf.dimensionless_confinement_indices({k: alpha[k] for k in names}, cov)
ratio = abs(1 + alpha["p_net"]) / fit.stderr[fit.names.index("p_net")]
print(f"completed alpha_R (ASSUMED) = {ind['alpha_R']:.2f} ± {ind['alpha_R_stderr']:.2f}")
for n_, v, s in zip(ind["names"], ind["value"], ind["stderr"]):
    print(f"  {n_:8s} {v:8.2f} ± {s:.2f}")
print(f"|1 + alpha_P| / sigma(alpha_P) = {ratio:.2f}  ->  "
      f"{'UNDETERMINED (mode B)' if ratio < 2 else 'determined'}")

completed alpha_R (ASSUMED) = 1.82 ± 0.37
  mu_rho     -13.27 ± 17.47
  mu_beta     -7.18 ± 9.07
  mu_nu       -2.21 ± 2.91
  mu_q       -20.29 ± 26.16
|1 + alpha_P| / sigma(alpha_P) = 0.87  ->  UNDETERMINED (mode B)


## Stage IV — engineering and dimensionless routes: when are they the same model?

Route A fits the engineering law and transforms it; route B fits
$\ln(\Omega_i\tau_E)$ directly on $\ln\rho_*$, $\ln\beta$, $\ln\nu_*$, $\ln q$.
On synthetic data that obey a dimensionless law exactly, with $q$ varied so the
engineering design has full rank, the two must agree to numerical precision,
provided the groups are defined identically.

In [ ]:
def synthetic(n_rows=200, seed=0, nu_with_q=False, collinear=False):
    r = np.random.default_rng(seed)
    n, T, B, R, q = (np.exp(r.uniform(-1, 1, n_rows)) for _ in range(5))
    if collinear:            # current and power locked together, as in an ohmic scan
        T = (B * R / q) ** 2 / (n * R**3) * np.exp(r.normal(0, 1e-3, n_rows))
    rho, beta = T**0.5 / (B * R), n * T / B**2
    nu = n * R / T**2
    omega_tau = rho**-2.7 * beta**-0.9 * nu**-0.01 * q**-3.0
    tau = omega_tau / B
    eng = {"i_p": B * R / q, "b_t": B, "p_net": n * T * R**3 / tau, "n_e": n, "r": R}
    nu_used = nu * q if nu_with_q else nu
    return tau, eng, omega_tau, {"rho_star": rho, "beta": beta, "nu_star": nu_used, "q": q}

def two_routes(**kw):
    tau_s, eng, omega_tau, groups = synthetic(**kw)
    g = np.arange(len(tau_s))
    route_a = conf.fit_confinement_scaling(tau_s, eng, g)
    keys = ["i_p", "b_t", "p_net", "n_e"]
    idx = [route_a.names.index(k) for k in keys]
    mu_a = conf.dimensionless_confinement_indices({k: route_a.exponents()[k] for k in keys},
                                                  np.asarray(route_a.cov)[np.ix_(idx, idx)])
    route_b = conf.fit_confinement_scaling(omega_tau, groups, g)
    return dict(zip(mu_a["names"], mu_a["value"])), route_b.exponents()

for label, kw in (("identical definitions", {}), ("nu_* defined with q (mode D)", {"nu_with_q": True})):
    a, b = two_routes(**kw)
    print(f"{label}:")
    for (ka, va), (kb, vb) in zip(a.items(), b.items()):
        print(f"   {ka:7s} route A {va:8.4f}   route B {vb:8.4f}   difference {vb - va:+.1e}")

identical definitions:
   mu_rho  route A  -2.7000   route B  -2.7000   difference +3.6e-15
   mu_beta route A  -0.9000   route B  -0.9000   difference +1.2e-14
   mu_nu   route A  -0.0100   route B  -0.0100   difference -1.1e-15
   mu_q    route A  -3.0000   route B  -3.0000   difference +2.0e-14
nu_* defined with q (mode D):
   mu_rho  route A  -2.7000   route B  -2.7000   difference +2.7e-15
   mu_beta route A  -0.9000   route B  -0.9000   difference +1.1e-14
   mu_nu   route A  -0.0100   route B  -0.0100   difference -7.8e-16
   mu_q    route A  -3.0000   route B  -2.9900   difference +1.0e-02


With identical definitions the routes agree to rounding. Defining $\nu_*$ with
a factor $q$ in one route and not the other moves $\mu_q$ by exactly $\mu_\nu$:
the model space is the same, the coordinates are not (mode **D**). That is a
definition mismatch, not a failure of either fit.

When the stored energy is tied to $I_p^2$, as in an ohmic scan, power follows
current and the engineering design loses a direction; the diagnostics say so
rather than the fit returning an arbitrary exponent (mode **A**):

In [ ]:
tau_c, eng_c, _, _ = synthetic(collinear=True)
diag = conf.assess_predictor_identifiability(eng_c)
print(f"locked I_p-P design: condition number {diag['condition_number']:.0f}, "
      f"largest VIF {max(diag['vif'].values()):.0f}")
pcs_c = conf.predictor_principal_directions(eng_c)
print("smallest singular value / largest:", f"{pcs_c['singular_values'][-1] / pcs_c['singular_values'][0]:.1e}")

locked I_p-P design: condition number 2711, largest VIF 1355629
smallest singular value / largest: 3.7e-04


## Stage V — VEST classified by failure mode

In [ ]:
summary = [
    ("A rank / identifiability", f"effective rank {pcs['effective_rank']}/3; weakest direction ~ ln P - ln I_p "
     f"(sv {pcs['singular_values'][-1]:.1f} vs {pcs['singular_values'][0]:.1f}); B_T spread follows R_geo "
     f"(B0R0 varies {b0r0.max() / b0r0.min() - 1:.0%})"),
    ("B ill-conditioned map", f"|1 + a_P|/sigma = {ratio:.2f} < 2: mu_rho, mu_beta, mu_nu undetermined"),
    ("C similarity constraint", "not testable on one machine (R fixed); the size exponent is completed, assumed"),
    ("D definitions", "W = W_mhd (magnetics EFIT), P = P_OH - dW/dt, n = Thomson line average; "
     "Thomson-inconsistent slices flagged (criteria v2, #1521)"),
]
print(pd.DataFrame(summary, columns=["mode", "VEST Tier A"]).to_string(index=False))

                    mode                                                                                                                      VEST Tier A
A rank / identifiability               effective rank 3/3; weakest direction ~ ln P - ln I_p (sv 2.0 vs 10.7); B_T spread follows R_geo (B0R0 varies 17%)
   B ill-conditioned map                                                                  |1 + a_P|/sigma = 0.87 < 2: mu_rho, mu_beta, mu_nu undetermined
 C similarity constraint                                                   not testable on one machine (R fixed); the size exponent is completed, assumed
           D definitions W = W_mhd (magnetics EFIT), P = P_OH - dW/dt, n = Thomson line average; Thomson-inconsistent slices flagged (criteria v2, #1521)


## Experimental design: which scan adds the most

The weak directions name the scans that would add information, in physical terms:

* **Loop voltage at fixed current** (the weakest direction, $\ln P - \ln I_p$):
  matched-$I_p$ discharges at different $V_{loop}$, e.g. by fuelling or wall
  conditioning, within one hardware block.
* **Toroidal field at matched $I_p$ and density**: today's $B_T$ spread follows
  $R_{geo}$ (plasma position) more than the TF current $B_0R_0$, which varies about
  half as much (the spreads and correlation printed in Stage I).
* **Density within one campaign block**: the two Tier A blocks differ in median
  density (printed in Stage I), so density and block are confounded.

None of these would let VEST separate Bohm from gyro-Bohm on its own: that needs
a $\rho_*$ scan at fixed $\beta$ and $\nu_*$, which requires a size or field
range a single small machine does not have.

## Not in this notebook yet (scoped follow-ups)

* **PCEIV** (principal-component errors-in-variables as used for NSTX), kept
  distinct from the partial-TLS `fit_confinement_scaling_odr`.
* **PLS** and a fuller PCR beyond the singular-direction diagnostic above.
* **Bayesian errors-in-variables** with correlated uncertainties.
* **ITPA20** as a documented published scaling.